# 03 · MVTec AD - Anomaly Detection + MLflow

**Approach:** Feature Extraction (ResNet18 backbone) + KNN anomaly scoring.

**Note:** No training loop → no per-epoch metrics.
MLflow logs: parameters, AUROC, the ROC curve and the score distribution.

| MLflow Experiment | `mvtec_anomaly_detection` |
|---|---|
| Metric | AUROC (higher → better separation of good from defect) |
| Server | `SQLite (mlflow.db)` |

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors
import mlflow, mlflow.pytorch

device = get_device()
DEVICE = device
ASSETS = 'cv/mvtec'


In [ ]:
import mlflow
import mlflow.pytorch

# MLflow - platform-safe SQLite (works on Windows + Mac)
mlflow.set_tracking_uri(get_mlflow_uri())
EXPERIMENT_NAME = "arkon-cv-mvtec"
RUN_NAME        = "patchcore_lite_v1"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow URI: {mlflow.get_tracking_uri()}')

MLFLOW_TAGS = {"dataset": "mvtec_ad", "task": "anomaly_detection", "method": "feature_extraction_knn", "framework": "pytorch", "architecture": "resnet18_layer4", "department": "Arkon_Components"}
print(f"MLflow URI : {MLFLOW_URI}")
print(f"Experiment : {EXPERIMENT_NAME}")

In [ ]:
DATA_DIR  = Path('../../../data/05_mvtec/raw')
MODEL_DIR = Path('../../../models/05_mvtec')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = Path('../../../models/checkpoints/mvtec')

PARAMS = {
    "category"       : "transistor",
    "img_size"       : 224,
    "batch_size"     : 32,
    "k_neighbors"    : 5,
    "backbone_layer" : "layer4",
    "num_workers"    : recommended_num_workers(device),
    "pin_memory"     : device.type == "cuda",
}
print(f'Config: {PARAMS}')


## 1. Dataset

In [ ]:
class MVTecDataset(Dataset):
    def __init__(self, root, split='train', transform=None, only_good=True):
        self.transform = transform; self.samples = []
        for defect_dir in sorted((root/split).iterdir()):
            if not defect_dir.is_dir(): continue
            is_good = defect_dir.name == 'good'
            if only_good and not is_good: continue
            label = 0 if is_good else 1
            for p in sorted(defect_dir.glob('*.png')):
                self.samples.append((p, label, defect_dir.name))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, label, defect = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label, defect

SZ   = PARAMS["img_size"]
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
tf   = transforms.Compose([transforms.Resize((SZ, SZ)),
                            transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set    = MVTecDataset(CAT_DIR, 'train', tf, only_good=True)
test_set     = MVTecDataset(CAT_DIR, 'test',  tf, only_good=False)
train_loader = DataLoader(train_set, PARAMS["batch_size"], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  PARAMS["batch_size"], shuffle=False, num_workers=0)
print(f"Train (good): {len(train_set)} | Test (all): {len(test_set)}")

## 2. Feature Extractor

In [ ]:
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
feature_extractor = nn.Sequential(*list(backbone.children())[:-2])  # up to layer4: [B,512,7,7]
feature_extractor = feature_extractor.to(DEVICE).eval()
total_params = sum(p.numel() for p in feature_extractor.parameters())
print(f"Backbone params: {total_params:,} (all frozen - zero training needed)")

## 3. Extract Features + MLflow Run

In [ ]:
ckpt = CheckpointManager(CKPT_DIR)
ckpt_name = f'mvtec_{PARAMS["category"]}_membank'

if ckpt.exists(ckpt_name):
    memory_bank, meta = ckpt.load_sklearn(ckpt_name)
    print(f'Loaded memory bank | extraction time: {meta.get("extract_time", "?")}')
else:
    with mlflow.start_run(run_name=RUN_NAME, tags=MLFLOW_TAGS) as run:
        mlflow.log_params(PARAMS)
        with Timer("Feature extraction") as t_extract:
                feats, labels, defects = [], [], []
                with torch.no_grad(), torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    for X, y, d in loader:
                        f = model(X.to(device)).mean(dim=[2,3])  # GAP → [B, 512]
                        feats.append(f.cpu().numpy())
                        labels.extend(y.tolist()); defects.extend(d)
                return np.vstack(feats), np.array(labels), defects
            
            with mlflow.start_run(run_name=RUN_NAME, tags=MLFLOW_TAGS) as run:
                mlflow.log_params(PARAMS)
                print(f"Run ID: {run.info.run_id}")
            
                # ── Feature extraction ────────────────────────────────────────────────────
                print("Extracting train (good) features → memory bank...")
                t0 = time.time()
                train_feats, _, _ = extract_features(train_loader, feature_extractor, DEVICE)
                print(f"Memory bank: {train_feats.shape} | {time.time()-t0:.1f}s")
            
                print("Extracting test features...")
                test_feats, test_labels, test_defects = extract_features(test_loader, feature_extractor, DEVICE)
                print(f"Test features: {test_feats.shape}")
            
                mlflow.log_metrics({
                    "memory_bank_size" : len(train_feats),
                    "test_size"        : len(test_feats),
                    "test_anomaly_rate": round(float(test_labels.mean()), 4),
                })
            
                # ── KNN anomaly scoring ───────────────────────────────────────────────────
                knn = NearestNeighbors(n_neighbors=PARAMS["knn_k"], metric=PARAMS["knn_metric"])
                knn.fit(train_feats)
                distances, _ = knn.kneighbors(test_feats)
                anomaly_scores = distances[:, 0]
            
                mlflow.log_metrics({
                    "score_min"  : round(float(anomaly_scores.min()), 4),
                    "score_max"  : round(float(anomaly_scores.max()), 4),
                    "score_mean" : round(float(anomaly_scores.mean()), 4),
                    "score_std"  : round(float(anomaly_scores.std()), 4),
                })
            
                # ── AUROC - key metric ────────────────────────────────────────────────────
                auroc = roc_auc_score(test_labels, anomaly_scores)
                mlflow.log_metric("auroc", round(auroc, 6))
                print(f"\n🎯 AUROC = {auroc:.4f}")
            
                # ── ROC curve → artifact ──────────────────────────────────────────────────
                fpr, tpr, _ = roc_curve(test_labels, anomaly_scores)
                fig, ax = plt.subplots(figsize=(7, 5))
                ax.plot(fpr, tpr, linewidth=2, label=f'ROC (AUROC={auroc:.4f})')
                ax.plot([0,1],[0,1],'--',color='gray',label='Random')
                ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
                ax.set_title(f'MVTec {CATEGORY.upper()} - Anomaly Detection ROC')
                ax.legend(); ax.grid(True)
                plt.tight_layout()
                with tempfile.TemporaryDirectory() as tmp:
                    p = os.path.join(tmp,"roc_curve.png")
                    plt.savefig(p, dpi=120); save_figure(fig, 'cv_mvtec_plot_1', subfolder='cv/mvtec')
plt.show(); mlflow.log_artifact(p, "plots")
            
                # ── Score distribution per defect type → artifact ────────────────────────
                import pandas as pd
                df_result = pd.DataFrame({'score': anomaly_scores, 'label': test_labels, 'defect': test_defects})
                fig2, ax2 = plt.subplots(figsize=(10, 4))
                for defect_type in sorted(df_result['defect'].unique()):
                    subset = df_result[df_result['defect']==defect_type]
                    ax2.hist(subset['score'], bins=15, alpha=0.6, label=defect_type)
                ax2.set_xlabel('Anomaly Score'); ax2.set_ylabel('Count')
                ax2.set_title(f'MVTec {CATEGORY.upper()} - Score Distribution per Defect')
                ax2.legend(); plt.tight_layout()
                with tempfile.TemporaryDirectory() as tmp:
                    p = os.path.join(tmp,"score_distribution.png")
                    plt.savefig(p, dpi=120); save_figure(fig, 'cv_mvtec_plot_1', subfolder='cv/mvtec')
plt.show(); mlflow.log_artifact(p, "plots")
            
                # ── Save the memory bank (numpy) for deployment ──────────────────────────
                import numpy as np
                bank_path = MODEL_DIR / f"memory_bank_{CATEGORY}.npy"
                np.save(bank_path, train_feats)
                mlflow.log_artifact(str(bank_path), "memory_bank")
            
                print(f"\n✅ Run complete | AUROC={auroc:.4f}")
                print(f"   Memory bank saved: {bank_path}")
        extract_time = t_extract.report()
        mlflow.log_param("extract_time", extract_time)
        ckpt.save_sklearn(memory_bank, ckpt_name,
                          metadata={"extract_time": extract_time})


## Summary

| Parameter | Value |
|---|---|
| MLflow Experiment | `mvtec_anomaly_detection` |
| Method | PatchCore-lite (ResNet18 Layer4 + KNN) |
| Key metric | **AUROC** |
| No training | feature extraction - zero training |
| Artifacts | ROC curve, score distribution, memory bank (.npy) |
| Change category | `PARAMS["category"]` = grid / metal_nut / screw / transistor |